# Practical9: Simple STRIPS / Goal-Stack Planning

**Objective:** Implement a simple planning problem using STRIPS / goal-stack planning.

In this practical, we will solve a very small **Robot Navigation Problem**.

A robot starts in **Room A** and must reach **Room C**.

The available route is:

```text
Room A  ---->  Room B  ---->  Room C
```

We will use simple STRIPS-style actions and a small goal-stack planner.

## 1. What Is a Planning Problem?

A planning problem has:

- an **initial state**: where we start,
- a **goal state**: what we want to achieve,
- **actions**: things the agent can do.

For our robot:

```text
Initial state: Robot is in Room A
Goal state:    Robot is in Room C
```

The robot can move:

```text
A → B
B → C
```


## 2. STRIPS Actions

In STRIPS, an action can have:

- **Preconditions**: what must already be true,
- **Add effects**: what becomes true after the action,
- **Delete effects**: what stops being true after the action.

For example:

```text
Action: Move A to B

Precondition: Robot is at A
Add:          Robot is at B
Delete:       Robot is at A
```


## 3. Define the Initial State and Goal

We will represent each fact using a simple string.


In [ ]:
initial_state = {"At_A"}

goal = "At_C"


### Understanding the code

```python
{"At_A"}
```

means the robot is currently in Room A.

The goal:

```python
"At_C"
```

means we want the robot to reach Room C.


## 4. Define the Actions

We only need two actions:

1. Move from A to B
2. Move from B to C


In [ ]:
actions = [
    {
        "name": "Move A to B",
        "pre": {"At_A"},
        "add": {"At_B"},
        "delete": {"At_A"}
    },
    {
        "name": "Move B to C",
        "pre": {"At_B"},
        "add": {"At_C"},
        "delete": {"At_B"}
    }
]


### Understanding one action

Look at:

```python
{
    "name": "Move A to B",
    "pre": {"At_A"},
    "add": {"At_B"},
    "delete": {"At_A"}
}
```

It means:

```text
IF the robot is at A,
THEN it can move to B.

After the move:
At_B becomes true.
At_A is removed.
```

That is the basic STRIPS idea.


## 5. Find an Action That Can Achieve a Goal

The planner must first ask:

> Which action can make my goal true?

For example, if the goal is `At_C`, the action **Move B to C** can achieve it because `At_C` is in its `add` set.


In [ ]:
def find_action(goal):
    for action in actions:
        if goal in action["add"]:
            return action

    return None


## 6. Goal-Stack Planner

The planner works backwards from the goal.

For `At_C`:

```text
Goal: At_C
   ↓
Need action: Move B to C
   ↓
But first we need: At_B
   ↓
Need action: Move A to B
   ↓
At_A is already true
```

Then the actions can be performed in the correct order:

```text
Move A to B
Move B to C
```


In [ ]:
def make_plan(goal, state, plan):
    # Goal is already true
    if goal in state:
        return True

    # Find an action that can achieve the goal
    action = find_action(goal)

    if action is None:
        return False

    # First achieve the action's preconditions
    for condition in action["pre"]:
        if not make_plan(condition, state, plan):
            return False

    # Perform the action
    state.difference_update(action["delete"])
    state.update(action["add"])
    plan.append(action["name"])

    return True


## 7. Understanding the Planner

First:

```python
if goal in state:
    return True
```

checks whether the goal is already true.

Then:

```python
action = find_action(goal)
```

finds an action that can achieve the goal.

Before performing that action, the planner checks its preconditions:

```python
for condition in action["pre"]:
    make_plan(condition, state, plan)
```

This is the **goal-stack idea**:

> If an action needs something first, that requirement becomes a new smaller goal.

Finally, the action changes the state:

```python
state.difference_update(action["delete"])
state.update(action["add"])
```

and its name is added to the plan.


## 8. Generate the Plan


In [ ]:
state = initial_state.copy()
plan = []

make_plan(goal, state, plan)

print("Plan:")
for step in plan:
    print(step)


### Expected Output

```text
Plan:
Move A to B
Move B to C
```

The planner first works backwards from the goal, then produces the actions in the order needed to reach Room C.


## 9. Quick Recap

In this practical:

- `initial_state` stores where the robot starts.
- `goal` stores what we want to achieve.
- Each STRIPS action has `pre`, `add`, and `delete`.
- `find_action()` finds an action that can achieve a goal.
- `make_plan()` works backwards through preconditions.
- The final plan moves the robot from Room A to Room C.

> **Main idea:** Goal-stack planning works backwards from the goal, finds the actions needed, and first satisfies their preconditions.
